## Ejercicio 2. Pipeline ETL con integración OLAP

## Introducción
En este ejercicio se implementa un pipeline ETL (Extracción, Transformación, Carga) completo 
que simula la integración de tres fuentes de datos heterogéneas —ERP, CRM y logs web— 
hacia un almacén de datos optimizado para OLAP con esquema estrella. 
El objetivo es comprender que la preparación de datos no es una tarea mecánica secundaria, 
sino la disciplina técnica que determina si la Minería de Datos descubrirá patrones reales 
o artefactos producidos por datos sucios.

## Objetivo
Implementar y documentar el pipeline ETL para integración OLAP, incluyendo las modificaciones solicitadas:
1. Extender el pipeline con una dimensión 'geografía' (ciudad, región) desde una cuarta fuente.
2. Implementar una regla para detectar outliers contextuales en la categoría Accesorios.
3. Diseñar un informe de calidad automático con resumen ejecutivo de máximo 5 líneas.
4. Reflexionar sobre la eficiencia del esquema estrella para OLAP frente a tablas planas.

## Insertar los imports

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

## Definición de la clase
A continuación se define la clase base del pipeline ETL proporcionada en el ejercicio.

In [2]:
class PipelineETLOLAP:
    """
    Pipeline ETL para integración OLAP.
    Simula extracción de fuentes heterogéneas, transformación rigurosa y carga en esquema estrella.
    Enfoque en decisiones técnicas fundamentales, no en herramientas específicas.
    """

    def __init__(self):
        self.fuente_erp = None      # Sistema ERP: transacciones operativas
        self.fuente_crm = None      # Sistema CRM: perfiles de clientes
        self.fuente_web = None      # Logs web: comportamiento digital
        self.datos_transformados = None
        self.almacen_datos = {
            "hechos_ventas": None,
            "dim_tiempo": None,
            "dim_producto": None,
            "dim_cliente": None,
            "dim_canal": None
        }

    def simular_fuentes_operacionales(self):
        """Genera datos sintéticos seguros que simulan fuentes heterogéneas reales"""
        print("="*70)
        print("SIMULANDO FUENTES OPERACIONALES HETEROGÉNEAS")
        print("="*70)

        # Fuente ERP: sistema transaccional con formato legacy
        erp_data = []
        productos_erp = [
            {"id": "P001", "nombre": "Laptop X1", "categoria_erp": "TECNOLOGIA", "precio": 1200.0},
            {"id": "P002", "nombre": "Monitor 24\"", "categoria_erp": "TECNOLOGIA", "precio": 300.0},
            {"id": "P003", "nombre": "Teclado Mecánico", "categoria_erp": "ACCESORIOS", "precio": 85.0},
            {"id": "P004", "nombre": "Mouse Inalámbrico", "categoria_erp": "ACCESORIOS", "precio": 45.0},
            {"id": "P005", "nombre": "Auriculares Bluetooth", "categoria_erp": "AUDIO", "precio": 120.0}
        ]

        for i in range(5000):
            producto = random.choice(productos_erp)
            fecha = datetime(2023, random.randint(1, 12), random.randint(1, 28))
            cantidad = random.randint(1, 5) if producto["categoria_erp"] != "TECNOLOGIA" else 1

            erp_data.append({
                "ID_TRANSACCION": f"T{i+1:06d}",
                "FECHA_VENTA": fecha.strftime("%d/%m/%Y"),
                "CODIGO_PRODUCTO": producto["id"],
                "CANTIDAD": cantidad,
                "PRECIO_UNITARIO": producto["precio"],
                "TOTAL_LINEA": round(cantidad * producto["precio"], 2),
                "ESTADO": random.choice(["COMPLETADA", "COMPLETADA", "COMPLETADA", "CANCELADA"])
            })

        self.fuente_erp = pd.DataFrame(erp_data)

        # Fuente CRM: sistema con esquema diferente y datos parciales
        crm_data = []
        clientes_crm = [
            {"id_cliente": "C1001", "nombre": "Tecnología Avanzada SA", "segmento": "EMPRESA"},
            {"id_cliente": "C1002", "nombre": "Soluciones Digitales Ltda", "segmento": "PYME"},
            {"id_cliente": "C1003", "nombre": "Juan Pérez", "segmento": "INDIVIDUAL"},
            {"id_cliente": "C1004", "nombre": "María García", "segmento": "INDIVIDUAL"},
            {"id_cliente": "C1005", "nombre": "Corporación Global S.A.", "segmento": "EMPRESA"}
        ]

        for i in range(5000):
            cliente = random.choice(clientes_crm)
            transaccion_id = f"T{i+1:06d}"

            crm_data.append({
                "TRANSACCION_ID": transaccion_id,
                "CLIENTE_ID": cliente["id_cliente"],
                "NOMBRE_CLIENTE": cliente["nombre"],
                "SEGMENTO_CLIENTE": cliente["segmento"],
                "CANAL_VENTA": random.choice(["TIENDA", "WEB", "TELEFONO", "TIENDA"])
            })

        self.fuente_crm = pd.DataFrame(crm_data)

        # Fuente Web: logs con formato JSON-like y campos opcionales
        web_data = []
        for i in range(5000):
            transaccion_id = f"T{i+1:06d}"
            sesion_id = f"S{i+1:06d}"

            web_data.append({
                "session_id": sesion_id,
                "transaction_id": transaccion_id,
                "pages_viewed": random.randint(3, 15),
                "time_on_site_minutes": round(random.uniform(2, 25), 1),
                "device_type": random.choice(["MOBILE", "DESKTOP", "TABLET", "MOBILE"]),
                "source_channel": random.choice(["ORGANIC", "PAID_SEARCH", "SOCIAL", "EMAIL", "DIRECT"])
            })

        self.fuente_web = pd.DataFrame(web_data)

        print(f"\n Fuentes simuladas con inconsistencias realistas:")
        print(f" • ERP: {len(self.fuente_erp):,} transacciones | Formato fecha: DD/MM/YYYY | Categorías en mayúsculas")
        print(f" • CRM: {len(self.fuente_crm):,} registros | Identificador: TRANSACCION_ID | Segmentos variados")
        print(f" • Web: {len(self.fuente_web):,} sesiones | Campos en inglés | Valores categóricos no estandarizados")
        print(f"\n Desafíos identificados para integración:")
        print(f" • Esquemas inconsistentes: ID_TRANSACCION vs TRANSACCION_ID vs transaction_id")
        print(f" • Formatos de fecha diferentes entre fuentes")
        print(f" • Categorías no alineadas (TECNOLOGIA vs TECHNOLOGY)")
        print(f" • Granularidad desigual (transacciones vs sesiones)")

    def fase_extraccion(self):
        """Extrae y normaliza datos de fuentes heterogéneas"""
        print("\n" + "="*70)
        print(" FASE EXTRACCIÓN: Normalización de esquemas heterogéneos")
        print("="*70)

        # Normalizar identificadores de transacción
        self.fuente_erp = self.fuente_erp.rename(columns={"ID_TRANSACCION": "transaction_id"})
        self.fuente_crm = self.fuente_crm.rename(columns={"TRANSACCION_ID": "transaction_id"})

        # Normalizar formato de fechas
        self.fuente_erp["fecha_venta"] = pd.to_datetime(self.fuente_erp["FECHA_VENTA"], format="%d/%m/%Y", errors="coerce")
        self.fuente_erp = self.fuente_erp.drop("FECHA_VENTA", axis=1)

        print(" Normalización completada:")
        print(" • Identificadores de transacción unificados bajo 'transaction_id'")
        print(" • Fechas convertidas a tipo datetime estándar")
        print(" • Columnas legacy eliminadas (FECHA_VENTA)")

    def fase_transformacion(self):
        """Limpia, enriquece y conforma datos para análisis OLAP"""
        print("\n" + "="*70)
        print(" FASE TRANSFORMACIÓN: Limpieza, enriquecimiento y conformación")
        print("="*70)

        # Paso 1: Limpieza de calidad
        print("\n1. LIMPIEZA DE CALIDAD:")
        datos_limpios = self.fuente_erp[
            (self.fuente_erp["ESTADO"] == "COMPLETADA") &
            (self.fuente_erp["TOTAL_LINEA"] > 0) &
            (self.fuente_erp["fecha_venta"].notna())
        ].copy()

        print(f" • Transacciones válidas: {len(datos_limpios):,} ({len(datos_limpios)/len(self.fuente_erp)*100:.1f}%)")
        print(f" • Eliminadas: {len(self.fuente_erp) - len(datos_limpios):,} (canceladas, montos cero o fechas inválidas)")

        # Paso 2: Enriquecimiento con catálogo maestro
        print("\n2. ENRIQUECIMIENTO SEMÁNTICO:")
        catalogo_maestro = {
            "P001": {"categoria": "Tecnología", "subcategoria": "Computación"},
            "P002": {"categoria": "Tecnología", "subcategoria": "Periféricos"},
            "P003": {"categoria": "Accesorios", "subcategoria": "Periféricos"},
            "P004": {"categoria": "Accesorios", "subcategoria": "Periféricos"},
            "P005": {"categoria": "Audio", "subcategoria": "Accesorios"}
        }

        datos_limpios["categoria_producto"] = datos_limpios["CODIGO_PRODUCTO"].map(
            lambda x: catalogo_maestro.get(x, {}).get("categoria", "Otros")
        )
        datos_limpios["subcategoria_producto"] = datos_limpios["CODIGO_PRODUCTO"].map(
            lambda x: catalogo_maestro.get(x, {}).get("subcategoria", "Otros")
        )
        print(" • Categorías estandarizadas usando catálogo maestro")
        print(" • Subcategorías añadidas para análisis granular")

        # Paso 3: Integración de fuentes
        print("\n3. INTEGRACIÓN DE FUENTES:")
        datos_integrados = datos_limpios.merge(
            self.fuente_crm[["transaction_id", "CLIENTE_ID", "SEGMENTO_CLIENTE", "CANAL_VENTA"]],
            on="transaction_id",
            how="left"
        ).merge(
            self.fuente_web[["transaction_id", "device_type", "source_channel"]],
            on="transaction_id",
            how="left"
        )

        # Manejo de nulos en uniones
        datos_integrados["SEGMENTO_CLIENTE"] = datos_integrados["SEGMENTO_CLIENTE"].fillna("Desconocido")
        datos_integrados["device_type"] = datos_integrados["device_type"].fillna("Desconocido")

        # Atributos derivados para análisis OLAP
        datos_integrados["mes"] = datos_integrados["fecha_venta"].dt.month
        datos_integrados["anio"] = datos_integrados["fecha_venta"].dt.year
        datos_integrados["ticket_alto"] = datos_integrados["TOTAL_LINEA"] > 500
        datos_integrados["canal_digital"] = datos_integrados["CANAL_VENTA"].isin(["WEB", "TELEFONO"])

        self.datos_transformados = datos_integrados
        print(" • Fuentes integradas con manejo explícito de nulos")
        print(" • Atributos derivados creados: mes, año, ticket_alto, canal_digital")

    def fase_carga(self):
        """Estructura datos en esquema estrella para OLAP y valida integridad"""
        print("\n" + "="*70)
        print(" FASE CARGA: Estructuración en esquema estrella y validación")
        print("="*70)

        # Crear tabla de hechos
        hechos_ventas = self.datos_transformados[[
            "transaction_id", "CLIENTE_ID", "fecha_venta", "mes", "anio",
            "CANTIDAD", "TOTAL_LINEA", "ticket_alto", "canal_digital"
        ]].copy()
        hechos_ventas = hechos_ventas.rename(columns={
            "CLIENTE_ID": "cliente_id",
            "CANTIDAD": "cantidad_productos",
            "TOTAL_LINEA": "monto_venta"
        })

        # Crear dimensiones
        dim_tiempo = hechos_ventas[["fecha_venta", "mes", "anio"]].drop_duplicates().reset_index(drop=True)
        dim_tiempo["tiempo_id"] = dim_tiempo.index + 1

        dim_producto = self.datos_transformados[[
            "CODIGO_PRODUCTO", "categoria_producto", "subcategoria_producto"
        ]].drop_duplicates().reset_index(drop=True)
        dim_producto = dim_producto.rename(columns={"CODIGO_PRODUCTO": "producto_id"})
        dim_producto["producto_sk"] = dim_producto.index + 1

        dim_cliente = self.datos_transformados[[
            "CLIENTE_ID", "SEGMENTO_CLIENTE"
        ]].drop_duplicates().reset_index(drop=True)
        dim_cliente = dim_cliente.rename(columns={"CLIENTE_ID": "cliente_id"})
        dim_cliente["cliente_sk"] = dim_cliente.index + 1

        dim_canal = self.datos_transformados[[
            "CANAL_VENTA", "device_type", "source_channel"
        ]].drop_duplicates().reset_index(drop=True)
        dim_canal["canal_sk"] = dim_canal.index + 1

        # Validación de integridad
        print("\n VALIDACIÓN DE INTEGRIDAD:")
        print(f" • Hechos de ventas: {len(hechos_ventas):,} registros")
        print(f" • Dimensión tiempo: {len(dim_tiempo):,} registros únicos")
        print(f" • Dimensión producto: {len(dim_producto):,} productos únicos")
        print(f" • Dimensión cliente: {len(dim_cliente):,} clientes únicos")
        print(f" • Dimensión canal: {len(dim_canal):,} combinaciones de canal únicas")

        # Métricas de calidad del pipeline
        tasa_retencion = len(hechos_ventas) / len(self.fuente_erp) * 100
        print(f"\n MÉTRICAS DE CALIDAD DEL PIPELINE:")
        print(f" • Tasa de retención: {tasa_retencion:.1f}% (transacciones válidas)")
        print(f" • Registros con datos completos: {len(hechos_ventas):,}")
        print(f" • Valores nulos manejados: {self.datos_transformados.isnull().sum().sum()}")

        # Almacenar en estructura de almacén de datos
        self.almacen_datos["hechos_ventas"] = hechos_ventas
        self.almacen_datos["dim_tiempo"] = dim_tiempo
        self.almacen_datos["dim_producto"] = dim_producto
        self.almacen_datos["dim_cliente"] = dim_cliente
        self.almacen_datos["dim_canal"] = dim_canal

        print("\n ALMACÉN DE DATOS LISTO PARA ANÁLISIS OLAP")
        print(" Estructura: Modelo estrella con 1 tabla de hechos + 4 dimensiones")
        print(" Optimizado para: Roll-up, Drill-down, Slice, Dice")

    def demostrar_valor_etl(self):
        """Compara escenarios con y sin pipeline ETL riguroso"""
        print("\n" + "="*70)
        print(" VALOR DEL PIPELINE ETL: Más allá de mover datos")
        print("="*70)

        print("\n[ESCENARIO SIN ETL RIGUROSO]")
        print(" • Analista carga datos crudos directamente en notebook")
        print(" • Encuentra errores: fechas inválidas, categorías inconsistentes, nulos")
        print(" • Dedica 6 horas a limpieza manual ad-hoc sin documentar")
        print(" • Modelo encuentra patrón: 'pan y leche' (obvio, no accionable)")
        print(" • Tiempo total: 8 horas | Valor generado: $0")

        print("\n[ESCENARIO CON PIPELINE ETL ESTRUCTURADO]")
        print(" • Pipeline automatiza limpieza, enriquecimiento y conformación")
        print(" • Analista recibe datos listos para análisis en 15 minutos")
        print(" • Modelo descubre: 'Auriculares Bluetooth + Laptop en clientes PYME por web'")
        print(" • Campaña dirigida aumenta ticket promedio 18% en segmento objetivo")
        print(" • Tiempo total: 3 horas (2h pipeline + 1h análisis) | Valor generado: $38,500")

        print("\n IMPACTO CUANTIFICABLE:")
        print(" • Inversión adicional en ETL: 2 horas × $75/h = $150")
        print(" • Retorno: $38,500 en ingresos adicionales")
        print(" • ROI del pipeline ETL: ($38,500 - $150) / $150 = 25,567%")

        print("\n LECCIÓN FUNDAMENTAL:")
        print(" El pipeline ETL no es un 'costo técnico'; es la inversión que multiplica")
        print(" el valor de todo el proceso KDD. Sin datos limpios y bien estructurados,")
        print(" incluso los algoritmos más sofisticados producen insights erróneos o irrelevantes.")
        print(" La calidad del input determina la utilidad del output: basura dentro, basura fuera.")

## Ejecución del pipeline ETL original
En esta sección se ejecuta la versión original del pipeline, respetando las fases 
de extracción, transformación y carga mostradas en el ejemplo base del PDF.

In [3]:
etl = PipelineETLOLAP()
etl.simular_fuentes_operacionales()
etl.fase_extraccion()
etl.fase_transformacion()
etl.fase_carga()
etl.demostrar_valor_etl()

SIMULANDO FUENTES OPERACIONALES HETEROGÉNEAS

 Fuentes simuladas con inconsistencias realistas:
 • ERP: 5,000 transacciones | Formato fecha: DD/MM/YYYY | Categorías en mayúsculas
 • CRM: 5,000 registros | Identificador: TRANSACCION_ID | Segmentos variados
 • Web: 5,000 sesiones | Campos en inglés | Valores categóricos no estandarizados

 Desafíos identificados para integración:
 • Esquemas inconsistentes: ID_TRANSACCION vs TRANSACCION_ID vs transaction_id
 • Formatos de fecha diferentes entre fuentes
 • Categorías no alineadas (TECNOLOGIA vs TECHNOLOGY)
 • Granularidad desigual (transacciones vs sesiones)

 FASE EXTRACCIÓN: Normalización de esquemas heterogéneos
 Normalización completada:
 • Identificadores de transacción unificados bajo 'transaction_id'
 • Fechas convertidas a tipo datetime estándar
 • Columnas legacy eliminadas (FECHA_VENTA)

 FASE TRANSFORMACIÓN: Limpieza, enriquecimiento y conformación

1. LIMPIEZA DE CALIDAD:
 • Transacciones válidas: 3,738 (74.8%)
 • Eliminadas: 

## Modificaciones realizadas

Con base en el apartado "Tu turno", se desarrolló una versión mejorada del pipeline para:
1. Incluir una dimensión 'geografía' (ciudad, región) integrada desde una cuarta fuente simulada.
2. Implementar detección de outliers contextuales: productos de categoría Accesorios con monto_venta > $500.
3. Generar un informe de calidad automático con resumen ejecutivo de máximo 5 líneas.
4. Reflexionar sobre la eficiencia del esquema estrella para OLAP frente a tablas planas.

In [4]:
class PipelineETLOLAPMejorado(PipelineETLOLAP):

    def simular_fuentes_operacionales(self):
        """Extiende la simulación agregando una cuarta fuente: base de datos de tiendas"""
        super().simular_fuentes_operacionales()

        # Fuente Geografía: base de datos de tiendas con ciudad y región
        ciudades = [
            {"ciudad": "Ciudad de México", "region": "Centro"},
            {"ciudad": "Guadalajara",      "region": "Occidente"},
            {"ciudad": "Monterrey",        "region": "Norte"},
            {"ciudad": "Puebla",           "region": "Centro"},
            {"ciudad": "Tijuana",          "region": "Norte"}
        ]
        geo_data = []
        for i in range(5000):
            ciudad_info = random.choice(ciudades)
            geo_data.append({
                "transaction_id": f"T{i+1:06d}",
                "ciudad": ciudad_info["ciudad"],
                "region": ciudad_info["region"]
            })
        self.fuente_geo = pd.DataFrame(geo_data)

        print(f" • Geo:  {len(self.fuente_geo):,} registros | Campos: ciudad, región")

    def fase_transformacion(self):
        """Extiende la transformación integrando la fuente geográfica"""
        super().fase_transformacion()

        # Integrar fuente geográfica
        self.datos_transformados = self.datos_transformados.merge(
            self.fuente_geo[["transaction_id", "ciudad", "region"]],
            on="transaction_id",
            how="left"
        )
        self.datos_transformados["ciudad"] = self.datos_transformados["ciudad"].fillna("Desconocida")
        self.datos_transformados["region"] = self.datos_transformados["region"].fillna("Desconocida")
        print(" • Dimensión geografía integrada: ciudad y región disponibles para análisis OLAP")

    def fase_carga(self):
        """Extiende la carga incluyendo la dimensión geografía en el esquema estrella"""
        super().fase_carga()

        dim_geografia = self.datos_transformados[
            ["ciudad", "region"]
        ].drop_duplicates().reset_index(drop=True)
        dim_geografia["geo_sk"] = dim_geografia.index + 1

        self.almacen_datos["dim_geografia"] = dim_geografia
        print(f" • Dimensión geografía: {len(dim_geografia):,} combinaciones únicas ciudad-región")
        print(" • Esquema estrella actualizado: 1 tabla de hechos + 5 dimensiones")

    def detectar_outliers_contextuales(self):
        """Detecta outliers en la categoría Accesorios con monto_venta > $500"""
        print("\n" + "="*70)
        print(" DETECCIÓN DE OUTLIERS CONTEXTUALES")
        print("="*70)

        accesorios = self.datos_transformados[
            self.datos_transformados["categoria_producto"] == "Accesorios"
        ].copy()

        outliers = accesorios[accesorios["TOTAL_LINEA"] > 500].copy()
        outliers["flag_revision"] = True

        print(f" • Total registros en categoría Accesorios: {len(accesorios):,}")
        print(f" • Outliers detectados (monto_venta > $500): {len(outliers):,}")
        if len(outliers) > 0:
            print(f" • Monto máximo detectado: ${outliers['TOTAL_LINEA'].max():,.2f}")
            print(f" • Transacciones marcadas para revisión:")
            print(outliers[["transaction_id", "CODIGO_PRODUCTO", "CANTIDAD", "TOTAL_LINEA"]].head(5).to_string(index=False))
        else:
            print(" • No se detectaron outliers en esta categoría.")

    def informe_calidad_automatico(self):
        """Genera resumen ejecutivo de calidad del pipeline en máximo 5 líneas"""
        print("\n" + "="*70)
        print(" INFORME DE CALIDAD — RESUMEN EJECUTIVO")
        print("="*70)

        total_entrada  = len(self.fuente_erp)
        total_salida   = len(self.almacen_datos["hechos_ventas"])
        tasa_retencion = total_salida / total_entrada * 100
        nulos_restantes = self.datos_transformados.isnull().sum().sum()
        dims = len(self.almacen_datos) - 1  # excluye hechos_ventas

        print(f" 1. Registros procesados: {total_entrada:,} entrada → {total_salida:,} salida (retención: {tasa_retencion:.1f}%).")
        print(f" 2. Valores nulos restantes en datos transformados: {nulos_restantes}.")
        print(f" 3. Esquema estrella generado con {dims} dimensiones (tiempo, producto, cliente, canal, geografía).")
        accesorios_df = self.datos_transformados[self.datos_transformados["categoria_producto"] == "Accesorios"]
        n_outliers = int((accesorios_df["TOTAL_LINEA"] > 500).sum())
        if n_outliers > 0:
            print(f" 4. Outliers contextuales en Accesorios: {n_outliers} registros detectados y marcados para revisión.")
        else:
            print(f" 4. Outliers contextuales en Accesorios: ninguno detectado con los datos actuales.")
        print(f" 5. Pipeline listo para EDA y modelado; se recomienda revisión manual de outliers antes del modelado.")

    def reflexion_esquema_estrella(self):
        """Reflexión: por qué el esquema estrella es más eficiente para OLAP que una tabla plana"""
        print("\n" + "="*70)
        print(" REFLEXIÓN: Esquema Estrella vs. Tabla Plana Desnormalizada")
        print("="*70)
        print("\n[TABLA PLANA DESNORMALIZADA]")
        print(" • Consulta: SELECT ciudad, categoria, SUM(monto) FROM tabla_plana GROUP BY ciudad, categoria")
        print("   → Escanea TODOS los campos de cada fila aunque solo se necesiten 3 columnas.")
        print("   → Mayor consumo de I/O y memoria; rendimiento degradado con volúmenes grandes.")
        print("\n[ESQUEMA ESTRELLA (OLAP)]")
        print(" • La tabla de hechos contiene solo métricas (monto, cantidad) y claves foráneas.")
        print(" • Las dimensiones están precalculadas y son pequeñas: joins rápidos y predecibles.")
        print(" • Consulta equivalente: JOIN dim_geografia + dim_producto → solo carga lo necesario.")
        print(" • Las herramientas OLAP (ej. PowerBI, Tableau) están optimizadas para este patrón.")
        print("\n CONCLUSIÓN:")
        print(" El esquema estrella reduce el escaneo de datos innecesarios, mejora el tiempo")
        print(" de respuesta de consultas analíticas y facilita el roll-up/drill-down sin reescribir SQL.")

## Ejecución del pipeline ETL mejorado
En esta sección se ejecuta la versión extendida del pipeline, incorporando la dimensión
geografía, la detección de outliers contextuales, el informe de calidad automático
y la reflexión sobre el esquema estrella.

In [5]:
# Instanciar el pipeline mejorado
etl_mejorado = PipelineETLOLAPMejorado()

# Ejecutar pipeline completo
etl_mejorado.simular_fuentes_operacionales()
etl_mejorado.fase_extraccion()
etl_mejorado.fase_transformacion()
etl_mejorado.fase_carga()
etl_mejorado.demostrar_valor_etl()

# Ejecutar las nuevas funcionalidades
etl_mejorado.detectar_outliers_contextuales()
etl_mejorado.informe_calidad_automatico()
etl_mejorado.reflexion_esquema_estrella()

SIMULANDO FUENTES OPERACIONALES HETEROGÉNEAS

 Fuentes simuladas con inconsistencias realistas:
 • ERP: 5,000 transacciones | Formato fecha: DD/MM/YYYY | Categorías en mayúsculas
 • CRM: 5,000 registros | Identificador: TRANSACCION_ID | Segmentos variados
 • Web: 5,000 sesiones | Campos en inglés | Valores categóricos no estandarizados

 Desafíos identificados para integración:
 • Esquemas inconsistentes: ID_TRANSACCION vs TRANSACCION_ID vs transaction_id
 • Formatos de fecha diferentes entre fuentes
 • Categorías no alineadas (TECNOLOGIA vs TECHNOLOGY)
 • Granularidad desigual (transacciones vs sesiones)
 • Geo:  5,000 registros | Campos: ciudad, región

 FASE EXTRACCIÓN: Normalización de esquemas heterogéneos
 Normalización completada:
 • Identificadores de transacción unificados bajo 'transaction_id'
 • Fechas convertidas a tipo datetime estándar
 • Columnas legacy eliminadas (FECHA_VENTA)

 FASE TRANSFORMACIÓN: Limpieza, enriquecimiento y conformación

1. LIMPIEZA DE CALIDAD:
 • Tr

## Desarrollo con evidencias

A lo largo del notebook se mostró la ejecución del pipeline original y del pipeline mejorado.
Las salidas generadas permiten comprobar la normalización de esquemas heterogéneos,
la limpieza y enriquecimiento de datos, la construcción del esquema estrella con cinco dimensiones
(incluyendo la nueva dimensión geografía), la detección de outliers contextuales en la categoría
Accesorios y la generación automática del informe de calidad ejecutivo.

## Conclusiones
El ejercicio demostró que un pipeline ETL riguroso es la columna vertebral que determina la calidad
de cualquier proyecto de Minería de Datos. La integración de fuentes heterogéneas requiere decisiones
técnicas precisas —unificación de identificadores, estandarización de formatos y enriquecimiento
semántico— que no pueden delegarse a la improvisación. La incorporación de la dimensión geografía
ilustra cómo el esquema estrella escala de forma natural sin rediseñar el modelo completo.
La detección de outliers contextuales y el informe de calidad automático refuerzan que
la calidad de datos no es un paso final sino un proceso continuo integrado en el pipeline.
Finalmente, la reflexión sobre el esquema estrella frente a tablas planas confirma que la
arquitectura elegida no es cosmética: impacta directamente el rendimiento de las consultas
analíticas y la capacidad de exploración multidimensional.

## Glosario
**ETL (Extracción, Transformación, Carga):** Proceso que integra datos de fuentes heterogéneas hacia un almacén analítico mediante limpieza, enriquecimiento y conformación.  
**OLAP (Online Analytical Processing):** Paradigma para consultas analíticas complejas sobre grandes volúmenes de datos históricos, basado en esquemas desnormalizados.  
**Esquema estrella:** Estructura de almacén de datos con una tabla de hechos central rodeada de tablas de dimensión; optimizada para roll-up, drill-down, slice y dice.  
**Outlier contextual:** Valor que resulta anómalo dentro de un subconjunto específico de datos (ej. precio elevado para una categoría de bajo costo habitual).  
**Dimensión:** Tabla de contexto cualitativo (tiempo, producto, cliente, canal, geografía) que rodea a la tabla de hechos en el modelo estrella.  
**Tasa de retención:** Porcentaje de registros que superan los filtros de calidad del pipeline y quedan disponibles para análisis.